# Nonlinear Optimizer Benchmark (Quick + Full presets)

This notebook runs the new nonlinear benchmark harness and writes outputs to `results/optimizer_benchmarks/` and `reports/optimizer_benchmarks/`.


In [ ]:
from pathlib import Path
import pandas as pd

from mpmgame.benchmark_suite import benchmark_problem_registry, run_benchmark_suite
from mpmgame.optimize_nonlinear import REQUIRED_ALGORITHMS

RESULTS_DIR = Path('../results/optimizer_benchmarks').resolve()
REPORTS_DIR = Path('../reports/optimizer_benchmarks').resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print('results:', RESULTS_DIR)
print('reports:', REPORTS_DIR)


## 1) Quick benchmark preset

Small, fast run across 3 toy chapter-2 instances with a compact algorithm set.


In [ ]:
quick_algorithms = ['Nelder-Mead', 'Powell', 'SLSQP', 'L-BFGS-B', 'differential_evolution']
quick_problems = benchmark_problem_registry('quick')

raw_quick, summary_quick = run_benchmark_suite(
    problem_specs=quick_problems,
    algorithms=quick_algorithms,
    n_restarts=5,
    timeout_per_run_sec=45,
    output_dir=RESULTS_DIR / 'quick',
    report_dir=REPORTS_DIR / 'quick',
    seed=7,
    show_progress=True,
    maxiter=150,
)

display(summary_quick.sort_values(['problem_id', 'best_feasible_objective']).head(20))


## 2) Full(er) benchmark preset

Broader algorithm set and longer per-run budget.


In [ ]:
full_algorithms = REQUIRED_ALGORITHMS
full_problems = benchmark_problem_registry('full')

raw_full, summary_full = run_benchmark_suite(
    problem_specs=full_problems,
    algorithms=full_algorithms,
    n_restarts=8,
    timeout_per_run_sec=60,
    algorithm_timeout_sec=8 * 60,
    output_dir=RESULTS_DIR / 'full',
    report_dir=REPORTS_DIR / 'full',
    seed=11,
    show_progress=True,
    maxiter=250,
)

display(summary_full.sort_values(['problem_id', 'best_feasible_objective']).head(30))


## 3) Compare nonlinear methods vs baselines


In [ ]:
def compare_against_baselines(raw_df):
    baselines = {'baseline_zero', 'baseline_projection', 'baseline_lp'}
    records = []
    for pid, d in raw_df.groupby('problem_id'):
        base_best = d[d['algorithm'].isin(baselines)].groupby('algorithm')['true_objective'].min()
        non_base = d[~d['algorithm'].isin(baselines)]
        best_non = non_base[non_base['feasible'] == True]['true_objective'].min() if not non_base.empty else float('inf')
        records.append({
            'problem_id': pid,
            'best_nonlinear': best_non,
            'best_baseline_projection': base_best.get('baseline_projection', float('inf')),
            'best_baseline_lp': base_best.get('baseline_lp', float('inf')),
            'best_baseline_zero': base_best.get('baseline_zero', float('inf')),
        })
    return pd.DataFrame(records)

comparison_quick = compare_against_baselines(raw_quick)
comparison_full = compare_against_baselines(raw_full)

print('Quick comparison')
display(comparison_quick)
print('Full comparison')
display(comparison_full)


## 4) Saved artifacts


In [ ]:
for sub in ['quick', 'full']:
    p = RESULTS_DIR / sub
    r = REPORTS_DIR / sub
    print(f'--- {sub} ---')
    print('results files:')
    for f in sorted(p.glob('*')):
        print(' ', f.name)
    print('report files:')
    for f in sorted(r.glob('*')):
        print(' ', f.name)
